# Non-Instruction Causal LLM Fine-Tuning on Bihar PDF

## Domain-Adaptive Continued Pretraining on a Complex Document

This notebook performs **non-instruction causal language model fine-tuning** on the Bihar.pdf document.

### What makes this different from a simple text PDF?
Bihar.pdf contains:
- Dense academic text (history, economics, governance)
- Tables with statistical data
- Figures and charts
- Footnotes and references

### Our approach:
1. We extract **text** (the primary training signal for causal LM)
2. We extract **tables** and convert them to text representations
3. We note images/figures but do NOT train on them (causal LM = text only)
4. We combine all textual content into a training corpus

### Pipeline
```text
Bihar PDF (complex document)
   ↓
Text extraction (PyMuPDF page-level)
   ↓
Table extraction (pdfplumber → text representation)
   ↓
Text cleaning and normalization
   ↓
Paragraph splitting + deduplication
   ↓
Hugging Face Dataset creation
   ↓
Tokenization + text packing into fixed blocks
   ↓
LoRA/QLoRA fine-tuning (TinyLlama)
   ↓
Validation loss + perplexity
   ↓
Adapter saving and reloading
   ↓
Text continuation inference
```

### Why Non-Instruction Fine-Tuning?
We feed the model **raw domain text** and it learns to predict the next token.
The model learns:
- Bihar-specific terminology (Magadha, Pataliputra, GSDP, bifurcation)
- Economic and governance language
- Historical writing style
- Statistical patterns and table descriptions

The model does **NOT** learn:
- How to answer questions
- How to follow instructions
- How to be a chatbot

(That would require Instruction Fine-Tuning in a separate step)

---
## Step 1: Install Required Libraries

We install:
- `pymupdf`: For PDF text and image extraction
- `pdfplumber`: For table extraction (works without Ghostscript unlike camelot)
- `datasets`: Hugging Face dataset creation
- `transformers`, `accelerate`: Model loading, tokenizer, Trainer
- `peft`: LoRA/QLoRA adapters
- `bitsandbytes`: 4-bit quantized model loading
- `sentencepiece`: Required by some tokenizers

In [19]:
!pip install -q -U "numpy<2.1" "scipy" "pymupdf" "datasets" "transformers" "accelerate" "peft" "bitsandbytes" "sentencepiece" "pillow<11.0" "pymupdf4llm" "docling" "pandas" "opencv-python" "tqdm" "pdfplumber"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 92.5 MB/s eta 0:00:00


---
## Step 2: Import All Libraries

We organize imports by category for clarity.

In [22]:
# ============================================================
# Standard Libraries
# ============================================================
import os
import re
import gc
import json
import math
import random
import warnings
import unicodedata
from dataclasses import dataclass, asdict
from typing import List, Dict, Any

warnings.filterwarnings("ignore")

# ============================================================
# Data Handling
# ============================================================
import numpy as np
import pandas as pd

# ============================================================
# PDF Processing
# ============================================================
import fitz          # PyMuPDF - for text + image extraction
import pdfplumber    # For table extraction

# ============================================================
# Hugging Face
# ============================================================
from datasets import Dataset, DatasetDict

# ============================================================
# PyTorch
# ============================================================
import torch

# ============================================================
# Transformers
# ============================================================
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)

# ============================================================
# PEFT (LoRA / QLoRA)
# ============================================================
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)

print("All imports successful!")

All imports successful!


---
## Step 3: Global Configuration

We put ALL parameters in a single dataclass.

**Why?**
- Easy to change one value and re-run the entire notebook
- Reproducibility: you can save/share the config
- No magic numbers scattered across cells

In [24]:
@dataclass
class Config:
    # ===== DATA =====
    # Upload Bihar.pdf to Colab and set this path
    pdf_path: str = "/content/Bihar.pdf"

    # ===== MODEL =====
    # TinyLlama: lightweight, good for learning/demo
    # For production, use Llama-2-7B, Mistral-7B, etc.
    model_name: str = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

    # ===== OUTPUT DIRECTORIES =====
    output_dir: str = "/content/bihar_lora_output"
    adapter_dir: str = "/content/bihar_lora_adapter"
    processed_data_dir: str = "/content/bihar_processed_data"

    # ===== TEXT PREPROCESSING =====
    # Paragraphs shorter than this are discarded (noise)
    min_chars_per_paragraph: int = 100
    # Each training block will have this many tokens
    block_size: int = 512

    # ===== TRAIN/EVAL SPLIT =====
    test_size: float = 0.10
    seed: int = 42

    # ===== LoRA PARAMETERS =====
    lora_r: int = 16           # Rank of LoRA matrices
    lora_alpha: int = 32       # Scaling factor
    lora_dropout: float = 0.05 # Dropout for regularization

    # ===== TRAINING PARAMETERS =====
    num_train_epochs: float = 3.0
    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    learning_rate: float = 2e-4
    warmup_steps: int = 10
    weight_decay: float = 0.01
    logging_steps: int = 5
    eval_steps: int = 20
    save_steps: int = 50
    save_total_limit: int = 2
    # Set max_steps=30 for quick demo, -1 for full training
    max_steps: int = -1


config = Config()
print(json.dumps(asdict(config), indent=2))

{
  "pdf_path": "/content/Bihar.pdf",
  "model_name": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
  "output_dir": "/content/bihar_lora_output",
  "adapter_dir": "/content/bihar_lora_adapter",
  "processed_data_dir": "/content/bihar_processed_data",
  "min_chars_per_paragraph": 100,
  "block_size": 512,
  "test_size": 0.1,
  "seed": 42,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "num_train_epochs": 3.0,
  "per_device_train_batch_size": 1,
  "per_device_eval_batch_size": 1,
  "gradient_accumulation_steps": 8,
  "learning_rate": 0.0002,
  "warmup_steps": 10,
  "weight_decay": 0.01,
  "logging_steps": 5,
  "eval_steps": 20,
  "save_steps": 50,
  "save_total_limit": 2,
  "max_steps": -1
}


In [25]:
config = Config()
os.makedirs(config.output_dir, exist_ok=True)
os.makedirs(config.adapter_dir, exist_ok=True)
os.makedirs(config.processed_data_dir, exist_ok=True)
print("Directories created.")

Directories created.


In [26]:
if not os.path.exists(config.pdf_path):
    print(f"PDF not found at: {config.pdf_path}")
else:
    print(f"PDF found: {config.pdf_path}")

PDF found: /content/Bihar.pdf


---
## Step 5: Extract Text from PDF (Page-Level)

PyMuPDF (`fitz`) extracts all text content from each page.

Bihar.pdf has 94 pages with:
- Academic paragraphs
- Footnotes
- Table captions and data
- Figure references

We extract everything as raw text first, then clean it.

In [27]:
def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    """
    Extract text from each page of a PDF using PyMuPDF.

    Returns a list of dicts with:
    - page: page number (1-indexed)
    - text: raw extracted text
    - char_count: length of text
    """
    pages = []
    with fitz.open(pdf_path) as doc:
        for page_index, page in enumerate(doc, start=1):
            text = page.get_text("text")
            text = text.strip() if text else ""
            if text:
                pages.append({
                    "page": page_index,
                    "text": text,
                    "char_count": len(text),
                })
    return pages

In [15]:
pdf_pages = extract_pdf_pages(config.pdf_path)

print(f"Total pages with extracted text: {len(pdf_pages)}")
print(f"\nPage-level character counts:")
for p in pdf_pages[:10]:  # Show first 10 pages
    print(f"  Page {p['page']:3d}: {p['char_count']:,} characters")
if len(pdf_pages) > 10:
    print(f"  ... and {len(pdf_pages) - 10} more pages")

total_chars = sum(p['char_count'] for p in pdf_pages)
print(f"\nTotal raw characters: {total_chars:,}")

Total pages with extracted text: 94

Page-level character counts:
  Page   1: 236 characters
  Page   2: 3,657 characters
  Page   3: 3,901 characters
  Page   4: 4,158 characters
  Page   5: 4,077 characters
  Page   6: 4,107 characters
  Page   7: 4,056 characters
  Page   8: 3,814 characters
  Page   9: 3,916 characters
  Page  10: 4,094 characters
  ... and 84 more pages

Total raw characters: 275,335


In [28]:
# Preview first page raw text
print("=" * 80)
print("RAW TEXT - Page 1 (first 1500 chars):")
print("=" * 80)
print(pdf_pages[0]["text"][:1500])

RAW TEXT - Page 1 (first 1500 chars):
1 
 
Bihar: What Went Wrong? And What Changed? 
 
Arnab Mukherji and Anjan Mukherji 
 
 
 
Working Paper No. 2012-107 
 
September 2012 
 
 
 
 
 
 
 
 
National Institute of Public Finance and Policy 
New Delhi 
http://www.nipfp.org.in


---
## Step 6: Extract Tables from PDF

Bihar.pdf has statistical tables (poverty data, HDI, crime rates, etc.).

We use `pdfplumber` to extract tables and convert them to text.

**Why convert tables to text?**
- Causal LM training requires text sequences
- The model can learn patterns like: "In 1981, Bihar's per capita income was Rs. 917"
- Table data converted to readable text adds domain knowledge

In [29]:
def extract_tables_as_text(pdf_path: str) -> List[Dict[str, Any]]:
    """
    Extract tables from PDF using pdfplumber and convert to text.

    Each table is converted to a string representation that can be
    included in the training corpus.
    """
    tables_data = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_index, page in enumerate(pdf.pages, start=1):
            page_tables = page.extract_tables()
            for t_index, table in enumerate(page_tables):
                if not table or len(table) < 2:
                    continue

                # Convert to DataFrame for clean text representation
                try:
                    # First row as header
                    headers = [str(h).strip() if h else f"Col_{i}"
                              for i, h in enumerate(table[0])]
                    df = pd.DataFrame(table[1:], columns=headers)

                    # Convert to readable text
                    table_text = df.to_string(index=False)

                    if len(table_text) > 50:  # Skip tiny/empty tables
                        tables_data.append({
                            "page": page_index,
                            "table_index": t_index + 1,
                            "rows": len(df),
                            "cols": len(df.columns),
                            "text": table_text,
                            "char_count": len(table_text),
                        })
                except Exception as e:
                    # Skip malformed tables
                    continue

    return tables_data

In [30]:
extracted_tables = extract_tables_as_text(config.pdf_path)

print(f"Total tables extracted: {len(extracted_tables)}")
print()

# Show first 3 tables
for i, t in enumerate(extracted_tables[:3]):
    print(f"Table {i+1} | Page {t['page']} | {t['rows']} rows x {t['cols']} cols")
    print(t['text'][:300])
    print("-" * 60)

Total tables extracted: 25

Table 1 | Page 68 | 6 rows x 18 cols
0.65\n0.55\n0.45\n0.35\n0.25\n1981 1983 1985 1987 1989 1991 1993 1995 1997 1999 Col_1 Col_2 Col_3 Col_4 Col_5 Col_6 Col_7 Col_8 Col_9 Col_10       Col_11 Col_12 Col_13 05 2007 2009 2011 Col_15 Col_16 Col_17
                                                                           None              
------------------------------------------------------------
Table 2 | Page 69 | 8 rows x 27 cols
Col_0 Col_1 Col_2 Col_3 Col_4 Col_5 Col_6 Col_7 Col_8 Col_9 Col_10 Col_11 Col_12 Col_13 Col_14 Col_15 Col_16 Col_17 Col_18 Col_19 Col_20 Col_21 Col_22 Col_23 Col_24 Col_25 Col_26
                                                               NaN                  NaN                         NaN      
------------------------------------------------------------
Table 3 | Page 69 | 15 rows x 26 cols
Col_0 Col_1 Col_2 Col_3 Col_4 Col_5 Col_6 Col_7 Col_8 Col_9 Col_10 Col_11 Col_12 Col_13 Col_14 Col_15 Col_16 Col_17 Col_18 Col_19 Col_20

---
## Step 7: Extract Image/Figure Metadata (For Reference Only)

**Important:** We do NOT train on images for causal LM.

But we log how many images exist for documentation purposes.
If you later want to do multimodal training, you would need a different approach.

In [31]:
def count_images(pdf_path: str) -> Dict[str, Any]:
    """Count embedded images in the PDF for documentation."""
    image_count = 0
    pages_with_images = []

    with fitz.open(pdf_path) as doc:
        for page_index, page in enumerate(doc, start=1):
            images = page.get_images(full=True)
            if images:
                image_count += len(images)
                pages_with_images.append(page_index)

    return {
        "total_images": image_count,
        "pages_with_images": pages_with_images,
    }

image_info = count_images(config.pdf_path)
print(f"Total embedded images/figures: {image_info['total_images']}")
print(f"Pages containing images: {image_info['pages_with_images'][:20]}")
print()
print("NOTE: Images are NOT used for text-based causal LM training.")
print("The model learns from text only. Figure captions (if present in text) ARE included.")

Total embedded images/figures: 36
Pages containing images: [51, 52, 54, 68, 71, 72, 73, 74, 76, 77, 78, 79, 82]

NOTE: Images are NOT used for text-based causal LM training.
The model learns from text only. Figure captions (if present in text) ARE included.


---
## Step 8: Text Cleaning

PDF extraction produces messy text. We need to clean it.

| Problem | Solution |
|---------|----------|
| Unicode oddities (ﬁ, ﬂ) | `unicodedata.normalize("NFKC")` |
| Zero-width spaces | Remove them |
| Hyphenated line breaks (gluconeogene-\nsis) | Join the word |
| Multiple spaces/tabs | Single space |
| Standalone page numbers | Remove |
| Excessive blank lines | Normalize to \n\n |

In [32]:
def clean_pdf_text(text: str) -> str:
    """
    Clean raw PDF-extracted text while preserving meaningful content.

    This handles common PDF extraction artifacts:
    - Unicode normalization
    - Hidden characters
    - Hyphenated line breaks
    - Multiple whitespace
    - Page numbers
    """
    # 1. Normalize Unicode (e.g., ﬁ → fi, ＡＭＰＫ → AMPK)
    text = unicodedata.normalize("NFKC", text)

    # 2. Remove zero-width and BOM characters
    text = text.replace("\u200b", "")  # zero-width space
    text = text.replace("\ufeff", "")  # byte order mark

    # 3. Fix hyphenated line breaks: "gover-\nnance" → "governance"
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # 4. Normalize multiple spaces/tabs to single space
    text = re.sub(r"[ \t]+", " ", text)

    # 5. Normalize excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # 6. Remove standalone page numbers (lines with only digits)
    text = re.sub(r"(?m)^\s*\d{1,3}\s*$", "", text)

    # 7. Split into paragraphs, clean each, reassemble
    paragraphs = re.split(r"\n\s*\n", text)
    cleaned_paragraphs = []

    for para in paragraphs:
        # Remove internal line breaks (PDF wrapping)
        para = re.sub(r"\n+", " ", para)
        # Normalize spaces
        para = re.sub(r"\s+", " ", para).strip()
        if para:
            cleaned_paragraphs.append(para)

    return "\n\n".join(cleaned_paragraphs)

In [33]:
# Clean all extracted pages
cleaned_pages = []
for page in pdf_pages:
    cleaned_text = clean_pdf_text(page["text"])
    cleaned_pages.append({
        "page": page["page"],
        "text": cleaned_text,
        "char_count": len(cleaned_text),
    })

print(f"Cleaned {len(cleaned_pages)} pages.")
print(f"Total cleaned characters: {sum(p['char_count'] for p in cleaned_pages):,}")
print()
print("Preview of cleaned Page 3:")
print("=" * 80)
print(cleaned_pages[2]["text"][:1200])

Cleaned 94 pages.
Total cleaned characters: 265,657

Preview of cleaned Page 3:
“For decades the sprawling state of Bihar, flat and scorching as a griddle, was something between a punch line and a cautionary tale, ... Criminals could count on the police for protection, not prosecution. Highwaymen ruled the shredded roads and kidnapping was one of the state’s most profitable businesses... Its government, led by politicians who used divisive identity politics to entrench their rule, was so corrupt that it required a newly coined phrase: the Jungle Raj.” Polgreen (2010)

This is an idea of Bihar that the majority of contemporary readers have encountered over and over again, especially after the 1980s. Polgreen (2010), however, in this article, goes on to describe not the decay of a once successful nation state, but rather a more remarkable change--the veritable signs of development and growth in a state that had once been considered a basket-case, or more politely, a failed state (see Fig

---
## Step 9: Split Into Paragraph Records

We split the cleaned text into individual paragraphs because:
- Easier to inspect and audit
- Can filter by length (remove noise)
- Can deduplicate repeated content
- Later, tokenization will pack these into fixed-size blocks anyway

In [34]:
def split_into_paragraph_records(
    cleaned_pages: List[Dict[str, Any]],
    min_chars: int = 100
) -> List[Dict[str, Any]]:
    """
    Split cleaned pages into individual paragraph records.

    - Filters out short paragraphs (likely noise/headers)
    - Deduplicates exact matches
    - Tracks source page for auditability
    """
    records = []
    seen = set()  # For deduplication

    for page in cleaned_pages:
        paragraphs = re.split(r"\n\s*\n", page["text"])

        for para_idx, paragraph in enumerate(paragraphs, start=1):
            paragraph = paragraph.strip()

            # Skip short paragraphs (noise, headers, page numbers)
            if len(paragraph) < min_chars:
                continue

            # Deduplicate
            key = re.sub(r"\s+", " ", paragraph.lower()).strip()
            if key in seen:
                continue
            seen.add(key)

            records.append({
                "text": paragraph,
                "source_page": page["page"],
                "paragraph_id": para_idx,
                "char_count": len(paragraph),
            })

    return records

In [35]:
# Create paragraph records from page text
paragraph_records = split_into_paragraph_records(
    cleaned_pages,
    min_chars=config.min_chars_per_paragraph
)

print(f"Total paragraph records from page text: {len(paragraph_records)}")
print()

# Also add table text as paragraph records
table_records = []
for t in extracted_tables:
    if t["char_count"] >= config.min_chars_per_paragraph:
        table_records.append({
            "text": f"Table data from page {t['page']}:\n{t['text']}",
            "source_page": t["page"],
            "paragraph_id": 0,  # 0 indicates table source
            "char_count": t["char_count"],
        })

print(f"Table records added: {len(table_records)}")

# Combine all records
all_records = paragraph_records + table_records
print(f"\nTotal training records: {len(all_records)}")
print(f"Total characters in corpus: {sum(r['char_count'] for r in all_records):,}")

Total paragraph records from page text: 418

Table records added: 25

Total training records: 443
Total characters in corpus: 290,640


In [36]:
# Preview some records
print("Sample records:")
for i in [0, 5, len(all_records)//2]:
    if i < len(all_records):
        print("=" * 80)
        print(f"Record {i} | Page {all_records[i]['source_page']} | {all_records[i]['char_count']} chars")
        print(all_records[i]["text"][:300])
        print()

Sample records:
Record 0 | Page 2 | 717 chars
Bihar as a political entity, either as a kingdom, or as a state within the republic of India, has its own identity from the time written records were available (Thapar 1966; Rangarajan 1992). Noted historian, Romila Thapar, describes the history of ancient India as the history of ancient Bihar. Many

Record 5 | Page 3 | 1013 chars
The Bihar section of the Essays on State Policies is organized in the following way: in Chapter 1 we provide a historical narrative of Bihar to provide context to much of its current state, and then we focus on its contemporary economy in the past three decades to better understand the moribund stat

Record 221 | Page 50 | 2010 chars
The Nitish Kumar-led government, whether by serendipity or design, appears to have achieved some success and can be expected to continue to deliver on these aspects with by now excellent experience of mobilizing and delivering public services. To keep himself aware of the concerns and

---
## Step 10: Save Processed Data (Auditability)

Always save intermediate data in real projects.
This helps with debugging, reproducibility, and compliance.

In [37]:
# Save processed corpus
corpus_path = os.path.join(config.processed_data_dir, "bihar_corpus.jsonl")

with open(corpus_path, "w", encoding="utf-8") as f:
    for record in all_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved {len(all_records)} records to: {corpus_path}")

Saved 443 records to: /content/bihar_processed_data/bihar_corpus.jsonl


---
## Step 11: Create Hugging Face Dataset + Train/Eval Split

We convert our records into a Hugging Face `Dataset` object.

Then we split into train and validation sets.

**Why keep a validation set?**
- Gives us validation loss (tells us if model is learning)
- Can compute perplexity (lower = better language understanding)
- Detects overfitting

In [38]:
if len(all_records) < 5:
    raise ValueError(
        "Corpus too small! Need at least 5 records. "
        "Check if PDF path is correct and extraction worked."
    )

# Create HF Dataset
text_dataset = Dataset.from_list(all_records)
print(f"Dataset: {text_dataset}")
print(f"\nSample entry:")
print(text_dataset[0])

Dataset: Dataset({
    features: ['text', 'source_page', 'paragraph_id', 'char_count'],
    num_rows: 443
})

Sample entry:
{'text': 'Bihar as a political entity, either as a kingdom, or as a state within the republic of India, has its own identity from the time written records were available (Thapar 1966; Rangarajan 1992). Noted historian, Romila Thapar, describes the history of ancient India as the history of ancient Bihar. Many achievements that India became renowned for, in education, governance, society, or religion, have their roots in Bihar. Significant achievements of Bihar in trade and economic engagement within the state and outside of the Indian sub-continent emerge from a past that appears to have left no living legacy in today’s Bihar--a past so alien as to be either simply forgotten or treated as being completely incredible.4', 'source_page': 2, 'paragraph_id': 5, 'char_count': 717}


In [39]:
# Train/validation split
split_dataset = text_dataset.train_test_split(
    test_size=config.test_size,
    seed=config.seed
)

raw_datasets = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"],
})

print(raw_datasets)
print(f"\nTrain samples: {len(raw_datasets['train'])}")
print(f"Validation samples: {len(raw_datasets['validation'])}")

DatasetDict({
    train: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 398
    })
    validation: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 45
    })
})

Train samples: 398
Validation samples: 45


---
## Step 12: Load Tokenizer

The tokenizer converts text → token IDs.

For causal LM:
```
"Bihar's economy" → [1, 350, 28742, 29915, 29879, 29871, 25378, 7586]
```

The model then learns: given tokens 1-7, predict token 8.

In [40]:
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

# Some models don't have a pad token - set it to EOS
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print(f"Tokenizer: {config.model_name}")
print(f"Vocab size: {len(tokenizer):,}")
print(f"Pad token: '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")
print(f"EOS token: '{tokenizer.eos_token}' (id={tokenizer.eos_token_id})")

Tokenizer: TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T
Vocab size: 32,000
Pad token: '</s>' (id=2)
EOS token: '</s>' (id=2)


---
## Step 13: Tokenize + Pack Into Fixed Blocks

Two-step process:
1. **Tokenize**: Convert all text to token IDs
2. **Pack/Group**: Concatenate all tokens, then split into blocks of `block_size` (512)

**Why pack?**
- Without packing: short paragraphs need lots of padding → wasted compute
- With packing: all blocks are exactly 512 tokens → efficient training

```
Para 1 (100 tokens) + Para 2 (200 tokens) + Para 3 (212 tokens) = Block 1 (512 tokens)
Para 4 (300 tokens) + Para 5 (212 tokens) = Block 2 (512 tokens)
```

In [41]:
def tokenize_function(examples):
    """Tokenize text without padding (packing handles it later)."""
    return tokenizer(examples["text"])


def group_texts(examples):
    """
    Pack tokenized sequences into fixed-size blocks.

    Steps:
    1. Concatenate all token sequences into one long sequence
    2. Split into chunks of block_size
    3. Drop the last incomplete chunk
    4. Labels = input_ids (model shifts internally for next-token prediction)
    """
    # Concatenate all sequences
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])

    # Drop remainder
    total_length = (total_length // config.block_size) * config.block_size

    if total_length == 0:
        return {k: [] for k in concatenated.keys()}

    # Split into blocks
    result = {
        k: [t[i:i + config.block_size] for i in range(0, total_length, config.block_size)]
        for k, t in concatenated.items()
    }

    # For causal LM: labels = input_ids
    result["labels"] = result["input_ids"].copy()
    return result

In [42]:
# Step 1: Tokenize
tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
    desc="Tokenizing",
)

print("After tokenization:")
print(tokenized_datasets)

Tokenizing:   0%|          | 0/398 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/45 [00:00<?, ? examples/s]

After tokenization:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 398
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 45
    })
})


In [43]:
# Step 2: Pack into fixed blocks
final_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    desc=f"Packing into {config.block_size}-token blocks",
)

print("\nAfter packing:")
print(final_datasets)
print(f"\nTrain blocks: {len(final_datasets['train'])}")
print(f"Validation blocks: {len(final_datasets['validation'])}")

if len(final_datasets['train']) == 0:
    raise ValueError("No training blocks created! PDF might be too small or block_size too large.")

Packing into 512-token blocks:   0%|          | 0/398 [00:00<?, ? examples/s]

Packing into 512-token blocks:   0%|          | 0/45 [00:00<?, ? examples/s]


After packing:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 141
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 12
    })
})

Train blocks: 141
Validation blocks: 12


In [44]:
# Verify a sample block
sample = final_datasets["train"][0]
print(f"Sample block - input_ids length: {len(sample['input_ids'])}")
print(f"Sample block - labels length: {len(sample['labels'])}")
print(f"\nDecoded preview (first 300 chars):")
print(tokenizer.decode(sample["input_ids"][:100]))

Sample block - input_ids length: 512
Sample block - labels length: 512

Decoded preview (first 300 chars):
<s> [2000-01, 2009-10] [2005-06,2009-10] Agriculture -0.20% 3.60% Industry 2.00% 6.00% Construction 23.70% 23.20% Services 8.40% 11.90% Source: Central Statistical Organ


---
## Step 14: Load Base Model with QLoRA Configuration

We load TinyLlama in **4-bit quantized** format:
- Uses ~1/4 the GPU memory
- Enables fine-tuning on free Colab GPUs (T4, 15GB)
- Industry standard for parameter-efficient fine-tuning

If CUDA is not available, we fall back to float32 (very slow on CPU).

In [46]:
use_cuda = torch.cuda.is_available()
print(f"CUDA available: {use_cuda}")
if use_cuda:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: Tesla T4
GPU Memory: 15.6 GB


In [47]:
# Clear memory before loading
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

if use_cuda:
    # 4-bit quantization config (QLoRA)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",           # NormalFloat4 quantization
        bnb_4bit_compute_dtype=torch.float16, # Compute in FP16
        bnb_4bit_use_double_quant=True,       # Double quantization saves more memory
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    # Required for stable gradients with quantized models
    base_model = prepare_model_for_kbit_training(base_model)
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

# Disable KV cache during training (saves memory, avoids warnings)
base_model.config.use_cache = False

print("Base model loaded successfully.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Base model loaded successfully.


---
## Step 15: Apply LoRA Adapters

**LoRA (Low-Rank Adaptation):**
- Instead of updating ALL 1.1B parameters, we add small trainable matrices
- Only ~1-3% of parameters are trained
- Much faster and cheaper than full fine-tuning
- The adapter can be saved separately (~30-50MB vs 2GB+ for full model)

**Target modules:**
- `q_proj, k_proj, v_proj, o_proj`: Attention layers
- `gate_proj, up_proj, down_proj`: Feed-forward layers

In [48]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


---
## Step 16: Data Collator

The data collator prepares mini-batches during training.

- `mlm=False` → We are doing **causal** LM (predict next token), NOT masked LM (BERT-style)

In [49]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal LM, not masked LM
)

---
## Step 17: Training Arguments

These control how training proceeds.

In [50]:
training_args = TrainingArguments(
    output_dir=config.output_dir,
    num_train_epochs=config.num_train_epochs,
    max_steps=config.max_steps,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    warmup_steps=config.warmup_steps,
    weight_decay=config.weight_decay,
    logging_steps=config.logging_steps,
    eval_strategy="steps",
    eval_steps=config.eval_steps,
    save_steps=config.save_steps,
    save_total_limit=config.save_total_limit,
    fp16=use_cuda,
    bf16=False,
    report_to="none",
    remove_unused_columns=False,
    seed=config.seed,
)

print("Training arguments configured.")

Training arguments configured.


---
## Step 18: Build Trainer and Start Training

In [51]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_datasets["train"],
    eval_dataset=final_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("Trainer ready. Starting training...")
print(f"Training blocks: {len(final_datasets['train'])}")
print(f"Validation blocks: {len(final_datasets['validation'])}")

Trainer ready. Starting training...
Training blocks: 141
Validation blocks: 12


In [52]:
# ============================================================
# TRAIN!
# ============================================================
train_result = trainer.train()

print("\n" + "=" * 60)
print("TRAINING COMPLETE!")
print("=" * 60)
print(f"Train loss: {train_result.training_loss:.4f}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss
20,2.128641,2.073144
40,2.032006,1.973163
54,1.919026,1.967684



TRAINING COMPLETE!
Train loss: 2.1334


---
## Step 19: Evaluate - Validation Loss & Perplexity

**Perplexity** = how "surprised" the model is by the validation text.
- Lower perplexity = model understands the domain better
- Formula: `perplexity = exp(validation_loss)`

In [53]:
eval_results = trainer.evaluate()

eval_loss = eval_results["eval_loss"]
perplexity = math.exp(eval_loss)

print(f"Validation Loss: {eval_loss:.4f}")
print(f"Perplexity: {perplexity:.2f}")
print()
print("Interpretation:")
print(f"  The model is, on average, choosing from ~{perplexity:.0f} equally likely next tokens.")
print(f"  Lower is better. Typical values for domain FT: 5-50.")

Training Loss,Validation Loss,Step
1.919026,1.967684,54


Validation Loss: 1.9677
Perplexity: 7.15

Interpretation:
  The model is, on average, choosing from ~7 equally likely next tokens.
  Lower is better. Typical values for domain FT: 5-50.


---
## Step 20: Save LoRA Adapter

We save only the adapter (small, ~30-50MB), not the full model.

To use it later:
1. Load the base model (TinyLlama)
2. Load the adapter on top
3. Generate text

In [54]:
# Save adapter + tokenizer
trainer.model.save_pretrained(config.adapter_dir)
tokenizer.save_pretrained(config.adapter_dir)

print(f"LoRA adapter saved to: {config.adapter_dir}")
print(f"\nSaved files:")
for f in os.listdir(config.adapter_dir):
    size = os.path.getsize(os.path.join(config.adapter_dir, f)) / 1024
    print(f"  {f} ({size:.1f} KB)")

LoRA adapter saved to: /content/bihar_lora_adapter

Saved files:
  README.md (5.1 KB)
  adapter_config.json (1.1 KB)
  tokenizer.json (3534.1 KB)
  adapter_model.safetensors (49319.9 KB)
  tokenizer_config.json (0.4 KB)


---
## Step 21: Reload Model + Adapter for Inference

This simulates how you would use the model in production:
1. Load fresh base model
2. Load saved LoRA adapter
3. Generate domain-specific continuations

In [55]:
# Clean up training objects
del trainer, model, base_model
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

print("Memory cleared. Reloading for inference...")

Memory cleared. Reloading for inference...


In [56]:
# Reload base model
if use_cuda:
    reload_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    inference_base = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=reload_config,
        device_map="auto",
        trust_remote_code=True,
    )
else:
    inference_base = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

# Load tokenizer from adapter directory
inference_tokenizer = AutoTokenizer.from_pretrained(config.adapter_dir, use_fast=True)
if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

# Load LoRA adapter on top of base model
inference_model = PeftModel.from_pretrained(inference_base, config.adapter_dir)
inference_model.eval()

print("Model + adapter loaded for inference!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model + adapter loaded for inference!


---
## Step 22: Text Continuation Inference

Since this is **non-instruction** fine-tuning, we give the model a text prompt
and it **continues** writing in the style/domain it learned.

Good prompts look like the beginning of a paragraph from Bihar.pdf.

**Bad prompt** (instruction-style): "What happened in Bihar after 2005?"

**Good prompt** (continuation-style): "After the 2005 elections in Bihar, Nitish Kumar"

In [57]:
def generate_continuation(prompt: str, max_new_tokens: int = 150) -> str:
    """
    Generate text continuation from a prompt.

    Parameters:
    - prompt: Starting text (should look like document text)
    - max_new_tokens: How many tokens to generate

    Returns:
    - Full text (prompt + generated continuation)
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = inference_tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,         # Sampling for diverse outputs
            temperature=0.7,        # Lower = more focused
            top_p=0.9,              # Nucleus sampling
            repetition_penalty=1.1, # Reduce repetition
            pad_token_id=inference_tokenizer.eos_token_id,
            eos_token_id=inference_tokenizer.eos_token_id,
        )

    return inference_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [58]:
# ============================================================
# TEST: Domain-specific text continuations
# ============================================================

test_prompts = [
    "Bihar as a political entity has its own identity from the time",
    "The Permanent Settlement Act of 1793 had devastating consequences for",
    "After the 2005 elections, the Nitish Kumar government focused on",
    "The bifurcation of Bihar into Bihar and Jharkhand in 2000 resulted in",
    "Bihar's per capita income relative to the national average",
]

for prompt in test_prompts:
    print("=" * 90)
    print(f"PROMPT: {prompt}")
    print("-" * 90)
    result = generate_continuation(prompt, max_new_tokens=150)
    # Show only the generated part
    continuation = result[len(prompt):]
    print(f"CONTINUATION: {continuation}")
    print()

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: Bihar as a political entity has its own identity from the time
------------------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


CONTINUATION:  of the Mauryas till today. The Mauryan Empire was a great state, that ruled over vast territories and spread its rule over the entirety of India. However, it had to contend with several competing states in its domains and often lost battles against them. Thus, there is no clear idea of what an ‘India’ would have looked like under Mauryan rule. During the Mauryan era, Bihar’s geography and climate were different from what they are today. As per Pandey (2008), “The early Mauryan period saw Bihar and Jharkhand being much smaller than they are today; there were only about 150 square kilometers

PROMPT: The Permanent Settlement Act of 1793 had devastating consequences for
------------------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


CONTINUATION:  the British Indian economy and society. By the end of the 18th century, there was no clear distinction between the ruling class and the non-ruling class; all were considered equal. Thus, the middle class, comprising a very small section of society, felt excluded from the benefits of the colonial economy. In addition, with the emergence of the Moghul Empire in India, trade in silks and other commodities began to be dominated by Muslim traders. Thus, in order to compete with them, the economy had to expand rapidly. In addition, the growth of industry was restricted. Apart from lack of capital, access to land also posed problems. The demand for land increased

PROMPT: After the 2005 elections, the Nitish Kumar government focused on
------------------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


CONTINUATION:  implementing a range of developmental schemes and programs. These included improving public health care facilities, making education more accessible to all, and improving sanitation and infrastructure for roads, railways, and ports. However, these programs were not only limited to governance but also in terms of the economy; the government was able to promote industrialization and agriculture. While Bihar’s economy continued to grow, it remained one of the lowest in India. The BJP government worked to ensure that its policies were implemented in Bihar and not just talked about. This led to an increase in government expenditure with little or no increase in revenue. Thus, even though Bihar grew faster than India, this growth was not sust

PROMPT: The bifurcation of Bihar into Bihar and Jharkhand in 2000 resulted in
------------------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


CONTINUATION:  substantial changes in the distribution of economic opportunity. Prior to this, Bihar had a significant advantage with regard to access to public services; the state government was able to provide more services at lower costs than the central government. In addition, Bihar had very strong public service delivery systems (PSS) and thus enjoyed high levels of trust from its citizens. However, with the division of Bihar into Bihar and Jharkhand, these advantages were lost. While the state government’s ability to provide services may have been reduced, it has also been significantly diminished as there is no longer a single PSS system for both states. This has led to an increase in the quality of life for people in the two states. Thus, while

PROMPT: Bihar's per capita income relative to the national average
------------------------------------------------------------------------------------------
CONTINUATION: , while Bihar’s has remained stagnant. The gap between Bihar an